In [2]:
import sys
sys.path.insert(0, '/home/allisond/nemo-curator-detached')

Activate GPU Acceleration,
Leverage NVIDIA cuML, cuDF

In [4]:
%load_ext cuml.accel
%load_ext cudf.pandas

The cuml.accel extension is already loaded. To reload it, use:
  %reload_ext cuml.accel
The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


Import Data

In [5]:
import pandas as pd

path = './peft-curation-with-sdg-70b/data/raw/splits/law-qa-train.jsonl'

# Load your initial dataset
dataset = pd.read_json(
    './peft-curation-with-sdg-70b/data/raw/splits/law-qa-train.jsonl', lines = True
)


In [6]:
dataset.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 19474 entries, 0 to 19473
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   file_name       19474 non-null  object
 1   id              19474 non-null  object
 2   title           19474 non-null  object
 3   question        19474 non-null  object
 4   question_score  19474 non-null  int64
 5   answer          19474 non-null  object
 6   answer_score    19474 non-null  int64
 7   tags            19474 non-null  object
dtypes: int64(2), object(6)
memory usage: 48.0+ MB


Clean Data

In [7]:
from bs4 import BeautifulSoup
import re

def clean_html(text):
    if not isinstance(text, str):
        return ""
    text = BeautifulSoup(text, "lxml").get_text()
    return re.sub(r"\s+", " ", text).strip()

import ftfy
from ftfy import TextFixerConfig

fix_config = TextFixerConfig()

def fix_unicode(text):
    if not isinstance(text, str):
        return ""
    return ftfy.fix_text(text, config=fix_config)

def word_count(text, min_words=50, max_words=500):
    words = text.strip().split()
    return min_words <= len(words) <= max_words

def filter_low_score(score, threshold=0):
    return score >= threshold


In [8]:
# dataset cleaning

def clean_filter_by_row(row):
    # Clean + fix each text field
    for field in ["title", "question", "answer"]:
        text = clean_html(row[field])
        text = fix_unicode(text)
        row[field] = text

    # Apply filters
    question_bool = word_count(row["question"]) and filter_low_score(float(row["question_score"]))
    answer_bool = word_count(row["answer"]) and filter_low_score(float(row["answer_score"]))

    return question_bool and answer_bool

def data_clean(dataset):
    dataset["keep"] = dataset.apply(clean_filter_by_row, axis=1)
    clean_dataset = dataset[dataset["keep"]].drop(columns=["keep"]).reset_index(drop=True)
    dataset.drop("keep", axis = 1, inplace = True)
    
    return clean_dataset


In [10]:
cleaned_dataset = data_clean(dataset)

In [11]:
cleaned_dataset.shape

(12244, 8)

Semantic Dedupe (cupy)

In [32]:
# semantic dedupe

def semantic_dedupe(dataset):
    dataset["text"] = dataset["title"] + dataset["question"] + dataset["answer"]

    # Step 1: Generate normalized sentence embeddings
    from sentence_transformers import SentenceTransformer
    import cupy as cp

    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(dataset["text"].tolist(), batch_size = 128, show_progress_bar = True, convert_to_numpy = True)
    embeddings_norm = cp.array(embeddings)/cp.linalg.norm(cp.array(embeddings), axis = 1, keepdims = True)

    # Step 2: Perform KMeans with cosine normalization (Euclidean KMeans on normalized vectors) 
    from sklearn.cluster import KMeans
    from sklearn.metrics.pairwise import cosine_similarity 

    n_clusters = 1000
    kmeans = KMeans(n_clusters = n_clusters, random_state = 1234, max_iter = 100)
    labels = kmeans.fit_predict(embeddings_norm)
    centroids = kmeans.cluster_centers_

    # Step 3: Deduplication logic (find points within eps_to_extract)
    eps_to_extract = 0.01
    dedup_indices = []
    for cluster_id in range(n_clusters):
        cluster_points = cp.where(cp.array(labels) == cluster_id)[0]
        if len(cluster_points) == 0:
            continue
        
        cluster_embeddings = embeddings_norm[cluster_points].get()
        centroid = centroids[cluster_id].reshape(1, -1)
        sims = cosine_similarity(cluster_embeddings, centroid).flatten()
        within_eps = cluster_points[sims >= (1 - eps_to_extract)]

        if len(within_eps) > 0:
            dedup_indices.extend(within_eps)
    
    # Step 4: Get deduplicated data
    mask = dataset.index.isin(dedup_indices) 
    kept_data = dataset[~mask]

    # step 5: drop the added column
    dataset.drop("text", axis = 1, inplace = True)
    kept_data.drop("text", axis = 1, inplace = True)

    return kept_data

Semantic Dedupe (numpy)

In [24]:
# semantic dedupe

def semantic_dedupe(dataset):
    dataset["text"] = dataset["title"] + dataset["question"] + dataset["answer"]

    # Step 1: Generate normalized sentence embeddings
    from sentence_transformers import SentenceTransformer
    import numpy as np

    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(dataset["text"].tolist(), batch_size = 128, show_progress_bar = True, convert_to_numpy = True, normalize_embeddings = True)

    # Step 2: Perform KMeans with cosine normalization (Euclidean KMeans on normalized vectors) 
    from sklearn.cluster import KMeans
    from sklearn.metrics.pairwise import cosine_similarity 

    n_clusters = 1000
    kmeans = KMeans(n_clusters = n_clusters, random_state = 1234, max_iter = 100)
    labels = kmeans.fit_predict(embeddings)
    centroids = kmeans.cluster_centers_

    # Step 3: Deduplication logic (find points within eps_to_extract)
    eps_to_extract = 0.01
    dedup_indices = []
    for cluster_id in range(n_clusters):
        cluster_points = np.where(labels == cluster_id)[0]
        
        if len(cluster_points) == 0:
            continue
        
        cluster_embeddings = embeddings[cluster_points]
        centroid = centroids[cluster_id].reshape(1, -1)
        sims = cosine_similarity(cluster_embeddings, centroid).flatten()
        within_eps = cluster_points[sims >= (1 - eps_to_extract)]

        if len(within_eps) > 0:
            dedup_indices.extend(within_eps)
    
    # Step 4: Get deduplicated data
    mask = dataset.index.isin(dedup_indices) 
    kept_data = dataset[~mask]

    # step 5: drop the added column
    dataset.drop("text", axis = 1, inplace = True)
    kept_data.drop("text", axis = 1, inplace = True)

    return kept_data

In [33]:
deduped_dataset = semantic_dedupe(cleaned_dataset)

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

In [34]:
deduped_dataset.shape

(12244, 8)

Synthetic Data Generation with Rewards

In [35]:
# Synthetic Data Generation

def SDG(dataset, sample_size_perc = 0.001, reward_threshold = 20):

    from openai import OpenAI
    from nemo_curator import OpenAIClient
    from nemo_curator.synthetic import NemotronGenerator
    import random

    openai_client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key="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"
    )
    client = OpenAIClient(openai_client)
    generator = NemotronGenerator(client)

    n_variants = 1
    sdg_model = "nvdev/nvidia/llama-3.1-nemotron-70b-instruct"
    sdg_model_kwargs = {
        "temperature": 0.2,
        "top_p": 0.7,
        "max_tokens": 1024,
        "seed": 1234
    }

    reward_model = "nvdev/nvidia/llama-3.1-nemotron-70b-reward"


    PROMPT_GENERATE_QUESTIONS_FROM_ANSWER = """TEXT:
    {document}

    Given the above text, generate exactly {n_openlines} questions that can be answered by the text. All questions must be answerable by the text and be relevant to the text.
    Do not directly reference the text in the questions.
    Every question should be a complete sentence and end with a question mark. There should be no other text besides the questions.
    Begin each question with `* ` and end each question with a newline character. Also, each question must be concise.
    Make sure to generate exactly {n_openlines} questions.
    """

    PROMPT_PARAPHRASE_TEXT = """TEXT:
        {document}

        Given the above text, paraphrase the text. Produce exactly {n_openlines} variants.
        There should be no other text besides the paraphrased text.
        The paraphrased text must be shorter than the original text. The paraphrased text must be factually correct and relevant to the original text.
        Begin each variant with `* ` and end each variant with a newline character.
        Make sure to generate exactly {n_openlines} variants.
        """

    output = []

    N = len(dataset)
    n = int(len(dataset) * sample_size_perc)
    sample_indices = random.sample(range(N), n)

    for idx in sample_indices:
        row = dataset.iloc[idx]
        question = row["question"]
        answer = row["answer"]

        gen_title = generator.generate_closed_qa_instructions(
            document=answer,
            n_openlines=n_variants,
            prompt_template=PROMPT_GENERATE_QUESTIONS_FROM_ANSWER,
            model=sdg_model,
            model_kwargs=sdg_model_kwargs,
        )

        gen_question = generator.generate_closed_qa_instructions(
                document=question,
                n_openlines=n_variants,
                prompt_template=PROMPT_PARAPHRASE_TEXT,
                model=sdg_model,
                model_kwargs=sdg_model_kwargs,
            )

        gen_answer = generator.generate_closed_qa_instructions(
            document=answer,
            n_openlines=n_variants,
            prompt_template=PROMPT_PARAPHRASE_TEXT,
            model=sdg_model,
            model_kwargs=sdg_model_kwargs,
        )

        messages = [
            {"role": "user", "content": f"{gen_title[0]}\n\n{gen_question[0]}"},
            {
                "role": "assistant",
                "content": f"{gen_answer[0]}",
            },
        ]

        rewards = client.query_reward_model(messages=messages, model=reward_model)

        if abs(rewards) <= reward_threshold:
            row_new = {
                "id": f"{row['id']}-synth-{n_variants}",
                "question": f"{gen_question[0]}",
                "answer": f"{gen_answer[0]}",
                "title": f"{gen_title[0]}",
                "file_name": "law-stackexchange-questions-answers.json.synth",
                "tags": f"{row['tags'] * n_variants}",
                "question_score": f"{rewards}",
                "answer_score": f"{rewards}"   
            }
        
            output.append(row_new)

    output_df = pd.DataFrame(output)

    return output_df

In [36]:
curated_dataset = SDG(deduped_dataset, sample_size_perc = 0.001, reward_threshold = 20)

In [37]:
curated_dataset.shape

(4, 8)

Synthetic Data Generation Pipeline

In [38]:
def SDG_Rewards_Pipeline(dataset, round_num, sample_size_perc = 0.001, reward_threshold = 20):

    cleaned_dataset = data_clean(dataset)
    deduped_dataset = semantic_dedupe(cleaned_dataset)
    print(f"After the initial curation, the dataset has {len(deduped_dataset)} records (originally {len(dataset)}).")
        
    dataset = deduped_dataset
    for num in range(round_num):
        
        sdg_dataset = SDG(dataset, sample_size_perc = sample_size_perc, reward_threshold = reward_threshold)
        dataset = pd.concat([dataset, sdg_dataset], axis = 0)
        deduped_dataset = semantic_dedupe(cleaned_dataset)

        print(f"After round {num + 1}, the dataset has {len(deduped_dataset)} records (originally {len(dataset)})")
        
    return deduped_dataset


In [39]:
curated_dataset = SDG_Rewards_Pipeline(dataset, 2, 0.001, 20)

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

After the initial curation, the dataset has 12244 records (originally 19474).


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

After round 1, the dataset has 12244 records (originally 12250)


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

After round 2, the dataset has 12244 records (originally 12254)


Output the Curated Dataset

In [40]:
curated_dataset.reset_index(drop = True, inplace = True)

In [41]:
curated_dataset.columns

Index(['file_name', 'id', 'title', 'question', 'question_score', 'answer',
       'answer_score', 'tags'],
      dtype='object')

In [44]:
# Specify the fields to include
fields_to_include = ['file_name', 'id', 'title', 'question', 'question_score', 'answer',
       'answer_score', 'tags']

# Output as JSONL
curated_dataset[fields_to_include].to_json(
    "curated_dataset.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)


/home/allisond/miniconda/envs/py311/lib/python3.11/site-packages/cudf/io/json.py:385: UserWarning: Using CPU via Pandas to write JSON dataset
  warnings.warn("Using CPU via Pandas to write JSON dataset")
